# **12. Comparación Final**

## **12.1. Librerias**

In [1]:
import os
import pandas as pd
from scipy import stats
import numpy as np
from scipy.stats import shapiro
import pandas as pd
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from scipy.stats import levene

from itertools import combinations

## **12.2.Cargando los Resultados**

In [2]:
BASE_DIR = "/home/guirlessa/Dl_Proyecto_Dengue/Modelos" 

MODELOS = ["LSTM", "CONVLSTM", "STGNN", "GRU", "Transformer","DDCRNN"]

def load_metrics(model_path):
    file_path = os.path.join(model_path, "metrics_stgnn.csv")

    # fallback si el nombre cambia por modelo
    if not os.path.exists(file_path):
        candidates = [f for f in os.listdir(model_path) if "metrics" in f and f.endswith(".csv")]
        if len(candidates) == 0:
            return None
        file_path = os.path.join(model_path, candidates[0])

    df = pd.read_csv(file_path)
    return df

all_metrics = []

for model in MODELOS:
    model_path = os.path.join(BASE_DIR, model)

    if not os.path.exists(model_path):
        print(f"No existe carpeta: {model_path}")
        continue

    df = load_metrics(model_path)

    if df is None:
        print(f"No metrics found for {model}")
        continue

    df["Model"] = model
    all_metrics.append(df)

# concatenar todo
metrics_df = pd.concat(all_metrics, ignore_index=True)
metrics_df.head()

,seed,RMSE,MAE,R2,Model
0,42,7.643459,4.205040,0.048068,LSTM
1,123,7.069677,3.966179,0.002728,LSTM
2,2024,7.050868,3.969128,0.008028,LSTM
3,42,0.677770,0.489266,0.438390,CONVLSTM
4,123,0.652954,0.469355,0.454739,CONVLSTM


## **12.3. Metricas Globales**

In [3]:
summary = metrics_df.groupby("Model").agg({
    "RMSE": ["mean", "std"],
    "MAE": ["mean", "std"],
    "R2": ["mean", "std"]
})

summary.columns = [
    "RMSE_mean", "RMSE_std",
    "MAE_mean", "MAE_std",
    "R2_mean", "R2_std"
]

summary = summary.reset_index()

summary

,Model,RMSE_mean,RMSE_std,MAE_mean,MAE_std,R2_mean,R2_std
0,CONVLSTM,0.658141,0.017618,0.473016,0.014764,0.457593,0.020778
1,DDCRNN,0.406530,0.002373,0.286674,0.003909,0.804324,0.006929
2,GRU,7.272854,0.315501,4.038056,0.092422,0.025908,0.083521
3,LSTM,7.254668,0.336835,4.046782,0.137063,0.019608,0.024789
4,STGNN,7.237096,0.089629,4.048328,0.060917,-0.003409,0.093412
5,Transformer,4.054371,0.304159,2.102692,0.131240,0.681954,0.067499


Los resultados obtenidos muestran que la arquitectura D-DCRNN alcanzó el mejor desempeño global entre los modelos evaluados, registrando los menores valores de error (RMSE = 0.4065; MAE = 0.2867) y el mayor coeficiente de determinación (R² = 0.8043). Asimismo, presentó la menor variabilidad entre ejecuciones, reflejada en desviaciones estándar considerablemente inferiores a las observadas en las demás arquitecturas. Aunque el Transformer constituyó la segunda alternativa con mejor capacidad predictiva (R² = 0.6820), sus errores fueron sustancialmente mayores y mostró una mayor sensibilidad a la inicialización. Por su parte, ConvLSTM alcanzó un desempeño intermedio, mientras que GRU, LSTM y STGNN exhibieron valores de R² cercanos a cero o negativos, indicando una limitada capacidad para representar la variabilidad de la incidencia.

## **12.4. Métricas por Semilla**

In [4]:
seed_analysis = metrics_df.groupby(["Model", "seed"]).agg({
    "RMSE": "mean",
    "MAE": "mean",
    "R2": "mean"
}).reset_index()

seed_analysis.head(15)

,Model,seed,RMSE,MAE,R2
0,CONVLSTM,42,0.677770,0.489266,0.438390
1,CONVLSTM,123,0.652954,0.469355,0.454739
2,CONVLSTM,2024,0.643700,0.460427,0.479651
3,DDCRNN,42,0.403859,0.283470,0.811572
4,DDCRNN,123,0.407338,0.285521,0.797765
5,DDCRNN,2024,0.408394,0.291029,0.803635
6,GRU,42,6.945436,3.931795,0.037472
7,GRU,123,7.574906,4.099742,0.103044
8,GRU,2024,7.298220,4.082630,-0.062792
9,LSTM,42,7.643459,4.205040,0.048068


Al analizar los resultados por semilla, la arquitectura D-DCRNN mantiene un comportamiento notablemente estable, con valores de RMSE comprendidos entre 0.4039 y 0.4084, MAE entre 0.2835 y 0.2910 y R² entre 0.7978 y 0.8116. Esta baja variabilidad indica que el proceso de entrenamiento converge sistemáticamente hacia soluciones muy similares independientemente de la inicialización aleatoria. En comparación con Transformer, cuyo desempeño promedio fue el segundo mejor entre los modelos evaluados (RMSE = 4.05 y R² = 0.682 en escala real), D-DCRNN alcanza una capacidad explicativa sustancialmente superior (R² ≈ 0.80 en escala logarítmica) junto con una dispersión entre semillas considerablemente menor (R² std = 0.0069 frente a 0.0675), lo que evidencia una mayor robustez y reproducibilidad de los resultados obtenidos.

La diferencia es aún más marcada respecto a las arquitecturas recurrentes tradicionales y al modelo STGNN. GRU, LSTM y STGNN presentan una elevada sensibilidad a la semilla de inicialización, reflejada en oscilaciones importantes de R² e incluso valores negativos en algunas ejecuciones, lo que indica dificultades para generalizar de manera consistente.

In [5]:
stability = metrics_df.groupby("Model").agg({
    "RMSE": "std",
    "MAE": "std",
    "R2": "std"
}).rename(columns={
    "RMSE": "RMSE_variability",
    "MAE": "MAE_variability",
    "R2": "R2_variability"
})

stability

,RMSE_variability,MAE_variability,R2_variability
Model,,,
CONVLSTM,0.017618,0.014764,0.020778
DDCRNN,0.002373,0.003909,0.006929
GRU,0.315501,0.092422,0.083521
LSTM,0.336835,0.137063,0.024789
STGNN,0.089629,0.060917,0.093412
Transformer,0.304159,0.131240,0.067499


Desde la perspectiva de la estabilidad entre ejecuciones, D-DCRNN presenta la menor variabilidad de todos los modelos evaluados, con desviaciones estándar de apenas 0.0024 en RMSE, 0.0039 en MAE y 0.0069 en R². En comparación, Transformer, aunque constituye el segundo mejor modelo en términos de desempeño predictivo, exhibe una sensibilidad considerablemente mayor a la semilla de inicialización, con variabilidades de 0.3042 en RMSE, 0.1312 en MAE y 0.0675 en R². Esto implica que D-DCRNN no solo alcanza mejores métricas promedio, sino que además genera resultados mucho más consistentes y reproducibles entre ejecuciones independientes.

La diferencia también es evidente frente al resto de arquitecturas. GRU y LSTM presentan las mayores fluctuaciones en RMSE y MAE, mientras que STGNN muestra la mayor variabilidad en R², reflejando una mayor dependencia de las condiciones iniciales del entrenamiento. Incluso ConvLSTM, que exhibe una estabilidad relativamente alta, mantiene niveles de variación superiores a los observados en D-DCRNN.

## **12.5. ANOVA**

El análisis de varianza (ANOVA) se utiliza para evaluar si las diferencias observadas en el desempeño entre los distintos modelos son estadísticamente significativas o si pueden atribuirse a la variabilidad inherente del proceso de entrenamiento. De esta manera, se garantiza una comparación rigurosa y objetiva bajo un mismo esquema experimental.

In [6]:
models = metrics_df["Model"].unique()

groups_rmse = [metrics_df[metrics_df["Model"] == m]["RMSE"].values for m in models]
groups_mae  = [metrics_df[metrics_df["Model"] == m]["MAE"].values for m in models]
groups_r2   = [metrics_df[metrics_df["Model"] == m]["R2"].values for m in models]

### **12.5.1. Supuestos**

Para aplicar ANOVA es necesario cumplir ciertos supuestos estadísticos, principalmente la normalidad de las distribuciones dentro de cada grupo, la homogeneidad de varianzas entre los grupos y la independencia de las observaciones. Estos requisitos son importantes porque aseguran que la comparación de medias entre modelos sea válida y que los resultados del test no estén sesgados por violaciones de las condiciones estadísticas que sustentan su inferencia.


#### **12.5.1.1. Normalidad**

In [7]:
print("TEST DE NORMALIDAD (Shapiro-Wilk)\n")

for model in models:
    data = metrics_df[metrics_df["Model"] == model]["RMSE"]

    stat, p = shapiro(data)

    print(f"🔹 {model} - RMSE")
    print(f"   W = {stat:.4f}, p = {p:.6f}")

    if p < 0.05:
        print("   No normal (rechaza H0)\n")
    else:
        print("   Normal (no se rechaza H0)\n")

TEST DE NORMALIDAD (Shapiro-Wilk)

🔹 LSTM - RMSE
   W = 0.7738, p = 0.053330
   Normal (no se rechaza H0)

🔹 CONVLSTM - RMSE
   W = 0.9350, p = 0.507554
   Normal (no se rechaza H0)

🔹 STGNN - RMSE
   W = 0.9631, p = 0.630615
   Normal (no se rechaza H0)

🔹 GRU - RMSE
   W = 0.9952, p = 0.866914
   Normal (no se rechaza H0)

🔹 Transformer - RMSE
   W = 0.7729, p = 0.051330
   Normal (no se rechaza H0)

🔹 DDCRNN - RMSE
   W = 0.9131, p = 0.428449
   Normal (no se rechaza H0)



Los resultados del test de Shapiro-Wilk indican que, para la métrica RMSE en todos los modelos evaluados, no se rechaza la hipótesis nula de normalidad (p > 0.05 en todos los casos). Esto sugiere que las distribuciones de error por modelo pueden considerarse aproximadamente normales.


#### **12.5.1.2. Homocedasticidad**

In [8]:
print("TEST DE LEVENE (Homogeneidad de varianzas)\n")

# RMSE
stat, p = levene(*groups_rmse)
print(f"RMSE -> W={stat:.4f}, p={p:.6f}")

# MAE
stat, p = levene(*groups_mae)
print(f"MAE  -> W={stat:.4f}, p={p:.6f}")

# R2
stat, p = levene(*groups_r2)
print(f"R2   -> W={stat:.4f}, p={p:.6f}")

TEST DE LEVENE (Homogeneidad de varianzas)

RMSE -> W=0.7228, p=0.619122
MAE  -> W=0.6608, p=0.659950
R2   -> W=0.7366, p=0.610239


Los resultados del test de Levene indican que no se rechaza la hipótesis nula de igualdad de varianzas para las métricas RMSE, MAE y R² (p > 0.05 en todos los casos). Esto sugiere que las varianzas entre los distintos modelos son homogéneas, cumpliendo así el supuesto de homocedasticidad requerido para la aplicación del ANOVA.

#### **12.5.1.3. Independencia de Observaciones**

La independencia de las observaciones se garantiza mediante el uso de distintas semillas aleatorias y particiones de validación cruzada, asegurando que cada ejecución del modelo sea estadísticamente independiente.

### **12.5.2. Aplicando ANOVA**

In [9]:
f_rmse, p_rmse = stats.f_oneway(*groups_rmse)

print(" ANOVA RMSE")
print(f"F-statistic: {f_rmse:.4f}")
print(f"p-value: {p_rmse}")

 ANOVA RMSE
F-statistic: 624.9052
p-value: 4.575935054137005e-14


El ANOVA aplicado a RMSE arroja un estadístico F = 624.9052 con p ≈ 4.575935054137005e-14, lo que implica el rechazo de la hipótesis nula de igualdad de medias entre modelos. Bajo este resultado, se concluye que las diferencias observadas en RMSE no pueden atribuirse a variabilidad muestral o aleatoria del proceso de entrenamiento, sino que existen efectos sistemáticos asociados a la arquitectura del modelo. En términos inferenciales, al menos un modelo pertenece a una distribución de error significativamente distinta, lo que valida estadísticamente la existencia de heterogeneidad estructural en el desempeño predictivo dentro del conjunto evaluado.

In [10]:
f_mae, p_mae = stats.f_oneway(*groups_mae)

print("\n ANOVA MAE")
print(f"F-statistic: {f_mae:.4f}")
print(f"p-value: {p_mae}")


 ANOVA MAE
F-statistic: 1211.5376
p-value: 8.733933185116829e-16


El ANOVA sobre MAE produce F = 1211.5376 con p ≈ 8.733933185116829e-16, lo que implica un rechazo de la hipótesis nula de igualdad de medias entre modelos. Este resultado indica que las diferencias en el error absoluto medio no son atribuibles a fluctuaciones aleatorias del proceso de entrenamiento, sino a diferencias sistemáticas inducidas por la arquitectura de cada modelo. En consecuencia, existe evidencia estadística robusta de heterogeneidad en el desempeño respecto a MAE, confirmando que al menos un subconjunto de modelos presenta un nivel de error significativamente distinto dentro del espacio experimental evaluado.

In [11]:
f_r2, p_r2 = stats.f_oneway(*groups_r2)

print("\n ANOVA R²")
print(f"F-statistic: {f_r2:.4f}")
print(f"p-value: {p_r2}")


 ANOVA R²
F-statistic: 112.1390
p-value: 1.2077206835506196e-09


El ANOVA aplicado al coeficiente de determinación (R²) arroja F = 112.1390 con p ≈ 1.2077206835506196e-09, lo que implica el rechazo de la hipótesis nula de igualdad de medias entre modelos para la capacidad explicativa. Aunque la magnitud del estadístico F es menor que en RMSE y MAE, el resultado sigue siendo estadísticamente altamente significativo, indicando que las diferencias observadas en R² no pueden explicarse por variabilidad aleatoria. En términos inferenciales, se confirma la existencia de heterogeneidad estructural en la capacidad de ajuste de los modelos, con al menos un subconjunto que presenta una varianza explicada significativamente distinta dentro del sistema evaluado.

## **12.6. TUKEY HSD**

Dado que el ANOVA evidenció la existencia de diferencias estadísticamente significativas entre las medias de desempeño de los modelos, y considerando que se verificó el cumplimiento de los supuestos de normalidad y homocedasticidad, se procede a aplicar la prueba post-hoc de Tukey HSD. Esta prueba permite identificar específicamente entre qué pares de modelos se presentan diferencias significativas, proporcionando una comparación múltiple controlada del error tipo I.

In [12]:
tukey_rmse = pairwise_tukeyhsd(
    endog=metrics_df["RMSE"],
    groups=metrics_df["Model"],
    alpha=0.05
)

print("TUKEY HSD - RMSE")
print(tukey_rmse)

TUKEY HSD - RMSE
    Multiple Comparison of Means - Tukey HSD, FWER=0.05    
 group1     group2   meandiff p-adj   lower   upper  reject
-----------------------------------------------------------
CONVLSTM      DDCRNN  -0.2516 0.7552 -0.8789  0.3756  False
CONVLSTM         GRU   6.6147    0.0  5.9875   7.242   True
CONVLSTM        LSTM   6.5965    0.0  5.9693  7.2238   True
CONVLSTM       STGNN    6.579    0.0  5.9517  7.2062   True
CONVLSTM Transformer   3.3962    0.0   2.769  4.0235   True
  DDCRNN         GRU   6.8663    0.0  6.2391  7.4936   True
  DDCRNN        LSTM   6.8481    0.0  6.2209  7.4754   True
  DDCRNN       STGNN   6.8306    0.0  6.2033  7.4578   True
  DDCRNN Transformer   3.6478    0.0  3.0206  4.2751   True
     GRU        LSTM  -0.0182    1.0 -0.6454  0.6091  False
     GRU       STGNN  -0.0358    1.0  -0.663  0.5915  False
     GRU Transformer  -3.2185    0.0 -3.8457 -2.5912   True
    LSTM       STGNN  -0.0176    1.0 -0.6448  0.6097  False
    LSTM Transformer  -

El análisis post hoc mediante la prueba de Tukey HSD para la métrica RMSE muestra que D-DCRNN presenta diferencias estadísticamente significativas respecto a Transformer, GRU, LSTM y STGNN (p < 0.001 en todos los casos), evidenciando un error de predicción significativamente menor. La diferencia media entre D-DCRNN y Transformer es de aproximadamente 3.65 unidades de RMSE, mientras que frente a GRU, LSTM y STGNN las diferencias superan las 6.8 unidades, lo que confirma una mejora sustancial en la precisión predictiva de la arquitectura propuesta.
Por otra parte, no se encontraron diferencias estadísticamente significativas entre D-DCRNN y ConvLSTM (p = 0.755), ya que el intervalo de confianza de la diferencia incluye el valor cero. Esto indica que, desde un punto de vista estrictamente estadístico, ambos modelos alcanzan niveles de error comparables. Sin embargo, considerando conjuntamente los resultados de desempeño promedio y estabilidad entre semillas, D-DCRNN es mejor al exhibir menor variabilidad y una capacidad explicativa superior.

In [13]:
tukey_mae = pairwise_tukeyhsd(
    endog=metrics_df["MAE"],
    groups=metrics_df["Model"],
    alpha=0.05
)

print("TUKEY HSD - MAE")
print(tukey_mae)

TUKEY HSD - MAE
    Multiple Comparison of Means - Tukey HSD, FWER=0.05    
 group1     group2   meandiff p-adj   lower   upper  reject
-----------------------------------------------------------
CONVLSTM      DDCRNN  -0.1863 0.1873 -0.4329  0.0602  False
CONVLSTM         GRU    3.565    0.0  3.3185  3.8116   True
CONVLSTM        LSTM   3.5738    0.0  3.3272  3.8203   True
CONVLSTM       STGNN   3.5753    0.0  3.3287  3.8219   True
CONVLSTM Transformer   1.6297    0.0  1.3831  1.8762   True
  DDCRNN         GRU   3.7514    0.0  3.5048  3.9979   True
  DDCRNN        LSTM   3.7601    0.0  3.5135  4.0067   True
  DDCRNN       STGNN   3.7617    0.0  3.5151  4.0082   True
  DDCRNN Transformer    1.816    0.0  1.5695  2.0626   True
     GRU        LSTM   0.0087    1.0 -0.2378  0.2553  False
     GRU       STGNN   0.0103    1.0 -0.2363  0.2568  False
     GRU Transformer  -1.9354    0.0 -2.1819 -1.6888   True
    LSTM       STGNN   0.0015    1.0  -0.245  0.2481  False
    LSTM Transformer  -1

Los resultados de la prueba de Tukey HSD para la métrica MAE muestran un patrón consistente con el observado en RMSE. D-DCRNN presenta diferencias estadísticamente significativas frente a Transformer, GRU, LSTM y STGNN (p < 0.001), obteniendo errores absolutos medios considerablemente menores. La diferencia media respecto a Transformer es de aproximadamente 1.82 unidades de MAE, mientras que frente a GRU, LSTM y STGNN supera las 3.75 unidades, lo que evidencia una mejora sustancial en la precisión de las predicciones.

Al igual que en RMSE, no se observan diferencias estadísticamente significativas entre D-DCRNN y ConvLSTM (p = 0.187), ya que el intervalo de confianza de la diferencia incluye el valor cero. Esto indica que ambos modelos alcanzan niveles de error absoluto similares desde una perspectiva estadística.

In [58]:
tukey_r2 = pairwise_tukeyhsd(
    endog=metrics_df["R2"],
    groups=metrics_df["Model"],
    alpha=0.05
)

print(" TUKEY HSD - R²")
print(tukey_r2)

 TUKEY HSD - R²
    Multiple Comparison of Means - Tukey HSD, FWER=0.05    
 group1     group2   meandiff p-adj   lower   upper  reject
-----------------------------------------------------------
CONVLSTM      DDCRNN   0.3467 0.0001  0.1831  0.5103   True
CONVLSTM         GRU  -0.4317    0.0 -0.5953 -0.2681   True
CONVLSTM        LSTM   -0.438    0.0 -0.6016 -0.2744   True
CONVLSTM       STGNN   -0.461    0.0 -0.6246 -0.2974   True
CONVLSTM Transformer   0.2244 0.0062  0.0608   0.388   True
  DDCRNN         GRU  -0.7784    0.0  -0.942 -0.6148   True
  DDCRNN        LSTM  -0.7847    0.0 -0.9483 -0.6211   True
  DDCRNN       STGNN  -0.8077    0.0 -0.9713 -0.6441   True
  DDCRNN Transformer  -0.1224 0.1948  -0.286  0.0412  False
     GRU        LSTM  -0.0063    1.0 -0.1699  0.1573  False
     GRU       STGNN  -0.0293 0.9889 -0.1929  0.1343  False
     GRU Transformer    0.656    0.0  0.4924  0.8197   True
    LSTM       STGNN   -0.023 0.9963 -0.1866  0.1406  False
    LSTM Transformer   0

Para la métrica R², la prueba de Tukey HSD muestra que D-DCRNN obtiene una capacidad explicativa significativamente superior a ConvLSTM, GRU, LSTM y STGNN (p < 0.001 en todos los casos). Las diferencias medias son particularmente amplias frente a GRU, LSTM y STGNN, con incrementos cercanos a 0.78–0.81 puntos de R², lo que indica una mejora sustancial en la proporción de variabilidad explicada por el modelo. Asimismo, D-DCRNN supera significativamente a ConvLSTM, con una diferencia media de 0.347 puntos de R².

Sin embargo, a diferencia de lo observado para RMSE y MAE, no se detectan diferencias estadísticamente significativas entre D-DCRNN y Transformer (p = 0.195). Aunque D-DCRNN presenta un R² medio superior, el intervalo de confianza de la diferencia incluye el valor cero, lo que sugiere que ambos modelos poseen una capacidad explicativa estadísticamente comparable. Este resultado posiciona a Transformer como la arquitectura más cercana a D-DCRNN en términos de explicación de la variabilidad de los datos.

## **12.7. Evaluando Robustez**

Para evaluar la robustez de los modelos, se analizaron las medias y desviaciones estándar de cada métrica de desempeño (RMSE, MAE y R²) a través de múltiples ejecuciones, con el fin de cuantificar tanto el rendimiento promedio como su variabilidad. Adicionalmente, se calculó el coeficiente de variación, lo que permitió estandarizar la dispersión relativa de los resultados y comparar la estabilidad entre modelos con diferentes escalas de error. Este enfoque facilita la identificación de aquellos modelos que no solo presentan buen desempeño promedio, sino también consistencia frente a variaciones en la inicialización y el proceso de entrenamiento, lo cual es fundamental para evaluar su robustez.

### **12.7.1. Medias & Desviaciones**

In [14]:
robustness = metrics_df.groupby("Model").agg({
    "RMSE": ["mean", "std"],
    "MAE": ["mean", "std"],
    "R2": ["mean", "std"]
})

robustness

RMSE                 MAE                  R2          
                 mean       std      mean       std      mean       std
Model                                                                  
CONVLSTM     0.658141  0.017618  0.473016  0.014764  0.457593  0.020778
DDCRNN       0.406530  0.002373  0.286674  0.003909  0.804324  0.006929
GRU          7.272854  0.315501  4.038056  0.092422  0.025908  0.083521
LSTM         7.254668  0.336835  4.046782  0.137063  0.019608  0.024789
STGNN        7.237096  0.089629  4.048328  0.060917 -0.003409  0.093412
Transformer  4.054371  0.304159  2.102692  0.131240  0.681954  0.067499

La comparación global de las arquitecturas evaluadas muestra que D-DCRNN obtuvo el mejor desempeño general, alcanzando los menores errores de predicción (RMSE = 0.4065; MAE = 0.2867) y la mayor capacidad explicativa (R² = 0.8043). Además, presentó la menor variabilidad entre ejecuciones (RMSE std = 0.0024; MAE std = 0.0039; R² std = 0.0069), lo que evidencia una alta estabilidad y reproducibilidad del proceso de entrenamiento. Estos resultados sugieren que la integración de mecanismos de difusión espacial junto con un grafo dinámico permite capturar de manera más efectiva las dependencias espacio-temporales presentes en los datos.

Transformer se posicionó como la segunda mejor arquitectura, con un R² de 0.6820, valor relativamente cercano al obtenido por D-DCRNN, aunque acompañado de errores considerablemente mayores (RMSE = 4.0544; MAE = 2.1027) y una variabilidad entre semillas notablemente superior. Por su parte, ConvLSTM alcanzó errores bajos en comparación con los modelos recurrentes tradicionales, pero presentó una capacidad explicativa sustancialmente inferior (R² = 0.4576). Finalmente, GRU, LSTM y STGNN exhibieron los desempeños más limitados, con valores de R² cercanos a cero o incluso negativos y errores elevados, indicando una capacidad reducida para representar adecuadamente la dinámica de la serie temporal.

### **12.7.2. Coeficiente de Variación**

In [15]:
cv_rmse = metrics_df.groupby("Model")["RMSE"].std() / metrics_df.groupby("Model")["RMSE"].mean()
cv_rmse

Model
CONVLSTM       0.026769
DDCRNN         0.005837
GRU            0.043381
LSTM           0.046430
STGNN          0.012385
Transformer    0.075020
Name: RMSE, dtype: float64

El modelo D-DCRNN presenta el menor coeficiente de variación (CV = 0.0058), indicando que sus resultados son extremadamente consistentes entre diferentes semillas de inicialización y que su desempeño apenas varía entre ejecuciones. Le siguen STGNN (CV = 0.0124) y ConvLSTM (CV = 0.0268), que también muestran niveles relativamente bajos de variabilidad. En contraste, Transformer exhibe el mayor coeficiente de variación (CV = 0.0750), lo que sugiere una mayor sensibilidad a las condiciones iniciales del entrenamiento y una menor reproducibilidad relativa de sus resultados. GRU (CV = 0.0434) y LSTM (CV = 0.0464) presentan igualmente niveles de variabilidad superiores a los observados en D-DCRNN.

## **12.8. Overfitting/underfitting**

En esta sección se analiza la presencia de **sobreajuste (overfitting) y subajuste (underfitting)** en los modelos evaluados a partir de sus curvas de aprendizaje. Para ello, se utiliza la diferencia entre la pérdida de validación y la pérdida de entrenamiento como indicador del grado de generalización del modelo, calculada a nivel de épocas y posteriormente agregada por semillas y particiones. Este enfoque permite cuantificar la brecha de generalización (generalization gap) y evaluar su estabilidad mediante medidas de tendencia central y dispersión, facilitando la comparación consistente del comportamiento de cada arquitectura bajo el mismo esquema experimental.


In [ ]:
BASE_DIR = "Modelos"

MODELOS = ["LSTM", "CONVLSTM", "STGNN", "GRU", "Transformer","DDCRNN"]

all_results = []

for model in MODELOS:

    path = os.path.join(BASE_DIR, model)

    file = [f for f in os.listdir(path) if "training_history" in f and f.endswith(".csv")]

    if len(file) == 0:
        print(f"No history found for {model}")
        continue

    history_path = os.path.join(path, file[0])
    history = pd.read_csv(history_path)

    history["gap"] = history["val_loss"] - history["train_loss"]

    summary_gap = history.groupby(["seed", "outer_fold"]).agg({
        "train_loss": "last",
        "val_loss": "last"
    }).reset_index()

    summary_gap["gap"] = summary_gap["val_loss"] - summary_gap["train_loss"]

    overfit_summary = summary_gap.groupby("seed").agg({
        "gap": ["mean", "std"]
    })

    overfit_summary.columns = ["gap_mean", "gap_std"]
    overfit_summary = overfit_summary.reset_index()

    overfit_summary["Model"] = model

    all_results.append(overfit_summary)

final_overfit = pd.concat(all_results, ignore_index=True)

final_overfit

,seed,gap_mean,gap_std,Model
0,42,0.301793,0.086989,LSTM
1,123,0.315937,0.101948,LSTM
2,2024,0.331889,0.119613,LSTM
3,42,0.085528,0.026504,CONVLSTM
4,123,0.120640,0.023259,CONVLSTM
5,2024,0.102050,0.016064,CONVLSTM
6,42,0.349910,0.155646,STGNN
7,123,0.344154,0.166403,STGNN
8,2024,0.342878,0.152486,STGNN
9,42,0.341189,0.106184,GRU


Los resultados muestran que Transformer y D-DCRNN presentan los menores valores de gap, con promedios cercanos a 0.02–0.03 en todas las semillas evaluadas. Esto sugiere que ambos modelos mantienen un equilibrio adecuado entre ajuste y generalización, logrando un desempeño consistente sobre datos no observados. En particular, D-DCRNN exhibe una estabilidad notable entre semillas, con valores de gap comprendidos entre 0.025 y 0.032, mientras que Transformer presenta los menores valores absolutos, aunque acompañado de una mayor variabilidad global observada previamente en sus métricas de evaluación.

Por otro lado, ConvLSTM muestra niveles moderados de separación entre entrenamiento y validación (gap ≈ 0.10), indicando una ligera pérdida de capacidad de generalización respecto a D-DCRNN y Transformer. En contraste, GRU, LSTM y STGNN registran los mayores valores de gap (entre 0.28 y 0.35), lo que evidencia una discrepancia más marcada entre el comportamiento observado durante el entrenamiento y el desempeño en validación.

In [17]:
model_summary = final_overfit.groupby("Model").agg({
    "gap_mean": ["mean", "std"],
    "gap_std": "mean"
})

model_summary.columns = [
    "gap_mean_mean",
    "gap_mean_std",
    "gap_std_mean"
]

model_summary = model_summary.reset_index()

print(model_summary)

         Model  gap_mean_mean  gap_mean_std  gap_std_mean
0     CONVLSTM       0.102739      0.017566      0.021942
1       DDCRNN       0.027982      0.003350      0.026675
2          GRU       0.310235      0.028894      0.080299
3         LSTM       0.316540      0.015057      0.102850
4        STGNN       0.345647      0.003746      0.158178
5  Transformer       0.024998      0.007128      0.021288


El análisis del generalization gap promedio muestra que las arquitecturas con mejor capacidad de generalización son Transformer y D-DCRNN, con diferencias medias entre entrenamiento y validación de 0.0250 y 0.0280, respectivamente. Estos valores son considerablemente inferiores a los observados en ConvLSTM (0.1027), GRU (0.3102), LSTM (0.3165) y STGNN (0.3456), lo que indica que tanto Transformer como D-DCRNN mantienen un comportamiento muy similar sobre datos de entrenamiento y validación, reduciendo el riesgo de sobreajuste.

Aunque Transformer presenta el menor gap promedio, D-DCRNN exhibe una estabilidad comparable entre semillas (gap_mean_std = 0.0034 frente a 0.0071), lo que sugiere una convergencia más consistente del proceso de entrenamiento. Por otra parte, GRU, LSTM y especialmente STGNN muestran brechas sustancialmente mayores, evidenciando una menor capacidad para generalizar el conocimiento aprendido durante el entrenamiento.

## **12.9. Conclusión**

Los resultados obtenidos permiten concluir que la arquitectura D-DCRNN constituye la alternativa más efectiva y robusta entre todos los modelos evaluados. En términos de precisión predictiva, alcanzó los menores errores (RMSE = 0.406 y MAE = 0.287) y la mayor capacidad explicativa (R² = 0.804), superando ampliamente a las arquitecturas recurrentes convencionales (LSTM y GRU), al modelo espacial STGNN y al ConvLSTM. Aunque el Transformer se posicionó como el benchmark de mejor desempeño, D-DCRNN logró una reducción aproximada del 90% en RMSE y del 86% en MAE, además de incrementar el R² desde 0.682 hasta 0.804, evidenciando una representación más precisa de la dinámica epidemiológica.

Adicionalmente, D-DCRNN mostró la mayor estabilidad entre ejecuciones, registrando la menor variabilidad entre semillas (RMSE std = 0.0024 y CV = 0.0058), lo que indica una convergencia altamente reproducible e independiente de la inicialización aleatoria. Los análisis complementarios de generalización también respaldan este comportamiento, con una brecha entrenamiento-validación reducida (gap ≈ 0.028), comparable a la del Transformer y sustancialmente menor que la observada en GRU, LSTM, STGNN y ConvLSTM. En conjunto, estos hallazgos sugieren que la incorporación simultánea de dependencias espaciales mediante grafos y dependencias temporales mediante mecanismos recurrentes permite a D-DCRNN capturar con mayor eficacia la estructura espacio-temporal del dengue, alcanzando un equilibrio superior entre precisión, capacidad explicativa, estabilidad y reproducibilidad respecto a todos los modelos benchmark evaluados.
